In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program that computes the Fast Fourier Transform (FFT) of a
  complex-valued 1-D signal. Given an input <code>signal</code> array containing
  <code>N</code> complex numbers stored as interleaved real/imaginary pairs,
  compute the discrete Fourier transform and store the result in the
  <code>spectrum</code> array. The FFT converts a time-domain signal into its
  frequency-domain representation using the formula: $$ X_k = \sum_{n=0}^{N-1}
  x_n \cdot e^{-j 2\pi kn / N} \quad \text{for } k = 0, 1, \ldots, N-1 $$ The
  FFT algorithm reduces the computational complexity from O(N²) to O(N log N) by
  exploiting symmetries in the twiddle factors.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries (cuFFT etc.) are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>spectrum</code> array</li>
  <li>The kernel must be entirely GPU-resident—no host-side FFT calls</li>
  <li>
    Both input and output use interleaved real/imaginary layout:
    <code>[real₀, imag₀, real₁, imag₁, ...]</code>
  </li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:  N = 4
        signal = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        (represents: [1+0j, 0+0j, 0+0j, 0+0j])

Output: spectrum = [1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0]
        (represents: [1+0j, 1+0j, 1+0j, 1+0j])
</pre>

<h2>Example 2:</h2>
<pre>
Input:  N = 2
        signal = [1.0, 0.0, 1.0, 0.0]
        (represents: [1+0j, 1+0j])

Output: spectrum = [2.0, 0.0, 0.0, 0.0]
        (represents: [2+0j, 0+0j])
</pre>

<h2>Constraints</h2>
<ul>
  <li><code>1 ≤ N ≤ 262,144</code></li>
  <li>All values are 32-bit floating point numbers</li>
  <li>Absolute error ≤ 1e-3 and relative error ≤ 1e-3</li>
  <li>Input and output arrays have length <code>2 × N</code></li>

  <li>Performance is measured with <code>N</code> = 262,144</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// signal and spectrum are device pointers
extern "C" void solve(const float* signal, float* spectrum, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# signal, spectrum are tensors on the GPU
@cute.jit
def solve(signal: cute.Tensor, spectrum: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# signal is a tensor on GPU
@jax.jit
def solve(signal: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# signal and spectrum are device pointers
@export
def solve(
    signal: UnsafePointer[Float32, MutExternalOrigin],
    spectrum: UnsafePointer[Float32, MutExternalOrigin],
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# signal and spectrum are device pointers
def solve(signal: torch.Tensor, spectrum: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# signal and spectrum are tensors on the GPU
def solve(signal: torch.Tensor, spectrum: torch.Tensor, N: int):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/hard/39_Fast_Fourier_transform/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
